# Module 3 - Class 4 Assignment: Feature Selection
**Khamidullokhon Abduvokhidov**

In [ ]:
# Load Telco data and prepare numeric target and charges.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [ ]:
# Combine numeric inputs with one-hot categorical inputs.
num = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']]
cat = pd.get_dummies(df[['Contract', 'PaymentMethod']], drop_first=True)
X = pd.concat([num, cat], axis=1)
y = df['Churn_bin']
print('X shape:', X.shape)
print('Features:', list(X.columns))

In [ ]:
# Measure each feature's mutual information with churn.
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)
mi_scores.round(4)

In [ ]:
# Visualize the feature information scores.
plt.figure(figsize=(8, 5))
mi_scores.plot(kind='barh', color='steelblue')
plt.title('Mutual information with Churn')
plt.xlabel('MI score')
plt.tight_layout()
plt.show()

In [ ]:
# Select the three strongest features with recursive elimination.
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=3)
rfe.fit(X_scaled, y)
result = pd.DataFrame({'feature': X.columns, 'kept': rfe.support_, 'rank': rfe.ranking_})
result.sort_values('rank')

In [ ]:
# Compare logistic-regression accuracy with every feature and only the selected three.
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
m_all = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print('All features:', round(accuracy_score(y_te, m_all.predict(X_te)), 4))
idx = np.where(rfe.support_)[0]
m_top = LogisticRegression(max_iter=1000).fit(X_tr[:, idx], y_tr)
print('Top 3:', round(accuracy_score(y_te, m_top.predict(X_te[:, idx])), 4))